# Structured Outputs

Structured outputs are useful when model responses are passed as inputs to other components of a system. 


## OpenAI API Structured Output

Historically, the OpenAI interface offered JSON output. However, using JSON output does not ensure adherence to a schema (data types, for example, may not be enforced). An alternative is to define the output schema using Pydantic.

A useful reference is the entry on [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs) from the API Documentation.

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
from pydantic import BaseModel
import os
os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

[Pydantic](https://docs.pydantic.dev/latest/) is a data validation library for Python. In Pydantic, we define [Models](https://docs.pydantic.dev/latest/concepts/models/) which are classes which inherit from [`BaseModel`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel) and define [`Fields`](https://docs.pydantic.dev/latest/api/fields/#pydantic.fields.Field) as annotated attributes.

In [ ]:
# BaseModel is a skeleton class, widely used in industry
# creates as basic class object CalendarEvent with 3 descriptors, error raised if it doesn't recieve it
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


In [ ]:
response = client.responses.parse(
    model=MODEL,
    input=[
        # admin prompt
        {"role": "system", "content": "Extract the event information."},
        # system prompt
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    # saving as class CalendarEvent
    text_format=CalendarEvent,
)
# output_parsed is a convenient way to output the text
event = response.output_parsed

In [ ]:
# event is no longer type string, but type CalendarEvents, can model_dump() to get a dictionary
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

## LangChain and Pydantic

LangChain offers capabilities to structure outputs following a specific schema. 

In the example below, we use LangChain to obtain a Joke object with a specific structure given by a Pydantic BaseModel.

In [ ]:
# LangChain is an aggregate of libraries, e.g. pdf extractors, sits on top of the low-level libraries

from langchain.chat_models import init_chat_model

# initiated using the same structure as prev, opens a model gateway, allows flexibility in which model to call
# change model_provider here to apply to all
llm = init_chat_model(MODEL, 
                      model_provider="openai",
                       base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                       default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
                      )

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field
# creating class Joke, Field() creates a description, can also apply many others, as metadata
# rating, is optional[int] defines as only integer, and Field() allows to set a defualt value

class Joke(BaseModel):
    setup: str=Field(description="The setup of the joke")
    punchline: str=Field(description="The punchline of the joke")
    rating: Optional[int] = Field(
        default=None, description="How funny the joke is, from 1 to 10"
    )

# saving the response with the structured output class Joke
structured_llm = llm.with_structured_output(Joke)

jk = structured_llm.invoke("Tell me a joke about cats")

In [8]:
response = llm.invoke('Hello model how are you?')

In [ ]:
# response is a LangChain object type AIMessage and obfuscating OpenAI object beneath
response.model_dump()

{'content': "Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?",
 'additional_kwargs': {'refusal': None},
 'response_metadata': {'token_usage': {'completion_tokens': 30,
   'prompt_tokens': 13,
   'total_tokens': 43,
   'completion_tokens_details': {'accepted_prediction_tokens': 0,
    'audio_tokens': 0,
    'reasoning_tokens': 0,
    'rejected_prediction_tokens': 0},
   'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
  'model_provider': 'openai',
  'model_name': 'gpt-4o-mini-2024-07-18',
  'system_fingerprint': 'fp_be7b4271a2',
  'id': 'chatcmpl-DvoKRea7Hlllw80PtxGTvfap97XhU',
  'service_tier': 'default',
  'finish_reason': 'stop',
  'logprobs': None},
 'type': 'ai',
 'name': None,
 'id': 'lc_run--019f0f71-a399-7553-8bc5-982765452545-0',
 'tool_calls': [],
 'invalid_tool_calls': [],
 'usage_metadata': {'input_tokens': 13,
  'output_tokens': 30,
  'total_tokens': 43,
  'input_token_details': 

In [12]:
jk.model_dump()

{'setup': 'Why did the cat sit on the computer?',
 'punchline': 'Because it wanted to keep an eye on the mouse!',
 'rating': 7}

## LangChain and TypedDict

We can also use a [typed dictionary or TypedDict](https://typing.python.org/en/latest/spec/typeddict.html) in Python to define the structure of our output. The keyword `Annotated[]` is used to wrap attributes of a typed dictionary.

In [13]:
# base Python also has a solution, TypedDict

from typing import Optional
from typing_extensions import Annotated, TypedDict

# Annotation includes data type, default value (if not is ...), and description
# Uses [] instead of () like Pydantic
class JokeDict(TypedDict):
    setup: Annotated[str, ..., "The setup of the joke"] # No default, with description
    punchline: Annotated[str, ..., "The punchline of the joke"] # No default, with description
    rating: Annotated[Optional[int], None, "How funny the joke is, from 1 to 10"] # Default of None, with description)

In [14]:
structured_llm_dict = llm.with_structured_output(JokeDict)
jk_dict = structured_llm_dict.invoke("Tell me a joke about dogs")


In [15]:
jk_dict

{'setup': 'Why did the dog sit in the shade?',
 'punchline': "Because he didn't want to become a hot dog!",
 'rating': 7}